In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pycocotools timm einops
!git clone https://github.com/facebookresearch/detr.git
%cd detr

In [ ]:
# ADD THIS RIGHT AFTER CLONING DETR AND BEFORE TRAINING:

# Patch the box_ops.py to remove assertions
import os
box_ops_path = '/kaggle/working/detr/util/box_ops.py'

with open(box_ops_path, 'r') as f:
    content = f.read()

# Comment out the assertion lines
content = content.replace(
    'assert (boxes1[:, 2:] >= boxes1[:, :2]).all()',
    '# assert (boxes1[:, 2:] >= boxes1[:, :2]).all()  # PATCHED FOR TERNARY QAT'
)
content = content.replace(
    'assert (boxes2[:, 2:] >= boxes2[:, :2]).all()',
    '# assert (boxes2[:, 2:] >= boxes2[:, :2]).all()  # PATCHED FOR TERNARY QAT'
)

with open(box_ops_path, 'w') as f:
    f.write(content)

print("✅ DETR box_ops.py patched - assertions disabled")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as funcy
import math

In [ ]:
class Rounding(torch.autograd.Function):
    @staticmethod
    def forward(ctx,x):
        return torch.round(x)
    @staticmethod
    def backward(ctx,grad_output):
        return grad_output

In [ ]:
class Ternary(nn.Module):
    def __init__(self,inn,out,bias=True):
        super().__init__()
        self.inn=inn
        self.out=out
        self.weight=nn.Parameter(torch.empty(out,inn,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.weight,a=math.sqrt(5))
        self.s=nn.Parameter(torch.tensor(0.1,dtype=torch.float32))
        self.register_buffer('qweight',torch.zeros(out,inn,dtype=torch.int8))
        if bias:
            self.bias=nn.Parameter(torch.zeros(out,dtype=torch.float32))
        else:
            self.bias=None
        with torch.no_grad():
            self.s.data=torch.mean(torch.abs(self.weight)).to(torch.float32).clamp(min=1e-5)
    def forward(self, x):
        if self.training:
            # BitNet b1.58 scaling logic
            sc = torch.mean(torch.abs(self.weight)).detach().clamp(min=1e-5)
            # Use Straight-Through Estimator
            w_quant = torch.clamp(torch.round(self.weight / sc), -1, 1)
            # This line ensures gradients flow through the latent weights
            ewe = sc * (w_quant + (self.weight / sc - (self.weight / sc).detach()))
        else:
            ewe = self.s * self.qweight.float()
        return funcy.linear(x, ewe, self.bias)
    def pack_for_inference(self):
        with torch.no_grad():
            sc=torch.mean(torch.abs(self.weight)).to(torch.float32).clamp(min=1e-5)
            wsc=self.weight/sc
            wcl=torch.clamp(wsc,-1.0,1.0)
            self.qweight.copy_(torch.round(wcl).to(torch.int8))
            self.s.data=sc

In [ ]:
def replace_with_scaled_ternary(mod):
    for name, module in list(mod.named_children()):
        if isinstance(module, nn.Linear):
            new_module = Ternary(
                module.in_features,
                module.out_features,
                bias=module.bias is not None
            )
            new_module.weight.data.copy_(module.weight.data)
            if module.bias is not None:
                new_module.bias.data.copy_(module.bias.data.to(torch.float32))
            setattr(mod, name, new_module)
        else:
            replace_with_scaled_ternary(module)
    return mod

In [ ]:
import torch
from argparse import Namespace
from models.detr import build
args = Namespace(
    dataset_file="coco",
    backbone="resnet50",
    num_queries=100,
    aux_loss=True,
    device="cuda",
    hidden_dim=256,
    dropout=0.1,
    nheads=8,
    enc_layers=6,
    dec_layers=6,
    dim_feedforward=2048,
    position_embedding="sine",
    dilation=False,
    normalize_before=False,
    lr_backbone=1e-5,
    masks=False,
    pre_norm=True,
    set_cost_class=1,
    set_cost_bbox=5,
    set_cost_giou=2,
    bbox_loss_coef=5,
    giou_loss_coef=2,
    eos_coef=0.1,
)
mod, criterion, postprocessors = build(args)
mod = replace_with_scaled_ternary(mod)
mod = mod.cuda()
print("✅ DETR-R50 successfully built with ScaledTernaryLinear (1.58-bit + shared FP16 scale)!")
print(f"   Total parameters: {sum(p.numel() for p in mod.parameters()):,}")
print(f"   Trainable parameters: {sum(p.numel() for p in mod.parameters() if p.requires_grad):,}")

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/detr')

In [ ]:
from datasets import build_dataset
from torch.utils.data import Subset
dataset_train = build_dataset(image_set='train', args=type('args', (), {'dataset_file': 'coco', 'coco_path': '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017','masks':False})())
dataset_train = Subset(dataset_train, range(10000))
sampler = torch.utils.data.RandomSampler(dataset_train)
data_loader_train = torch.utils.data.DataLoader(
    dataset_train, batch_size=1, sampler=sampler,
    collate_fn=collate_fn, num_workers=0
)
dataset_val = build_dataset(image_set='val', args=type('args', (), {'dataset_file': 'coco', 'coco_path': '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017','masks':False})())
data_loader_val = torch.utils.data.DataLoader(dataset_val, batch_size=1, shuffle=False, collate_fn=collate_fn)

In [ ]:
param_dicts = [
    {
        "params": [p for n, p in mod.named_parameters() if "backbone" not in n and p.requires_grad],
        "lr": 2e-5,      # Reduced from 1e-4
    },
    {
        "params": [p for n, p in mod.named_parameters() if "backbone" in n and p.requires_grad],
        "lr": 2e-6,      # Very slow for backbone to preserve features
    },
]
optimizer = torch.optim.AdamW(param_dicts, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
def sanitize_boxes_cxcywh(boxes):
    if boxes.numel() == 0:
        return boxes.view(0, 4)
    original_shape = boxes.shape
    needs_batch_reshape = False
    if boxes.dim() == 3:
        B, N, _ = boxes.shape
        boxes = boxes.reshape(-1, boxes.shape[-1])
        needs_batch_reshape = True
    elif boxes.dim() == 1:
        boxes = boxes.reshape(1, -1)
    boxes = torch.nan_to_num(boxes, nan=0.5, posinf=1.0, neginf=0.0)
    cx = boxes[:, 0].clamp(0.0, 1.0)
    cy = boxes[:, 1].clamp(0.0, 1.0)
    w = boxes[:, 2].clamp(1e-6, 1.0)
    h = boxes[:, 3].clamp(1e-6, 1.0)
    x1 = cx - w / 2
    y1 = cy - h / 2
    x2 = cx + w / 2
    y2 = cy + h / 2
    x1 = x1.clamp(0.0, 1.0)
    y1 = y1.clamp(0.0, 1.0)
    x2 = x2.clamp(0.0, 1.0)
    y2 = y2.clamp(0.0, 1.0)
    eps = 1e-4
    x2 = torch.maximum(x2, x1 + eps)
    y2 = torch.maximum(y2, y1 + eps)
    cx_new = (x1 + x2) / 2
    cy_new = (y1 + y2) / 2
    w_new = (x2 - x1).clamp(min=1e-6)
    h_new = (y2 - y1).clamp(min=1e-6)
    sanitized = torch.stack([cx_new, cy_new, w_new, h_new], dim=1)
    sanitized = torch.nan_to_num(sanitized, nan=0.5, posinf=1.0, neginf=0.0)
    if boxes.shape[1] > 4:
        sanitized = torch.cat([sanitized, boxes[:, 4:]], dim=1)
    if needs_batch_reshape:
        sanitized = sanitized.view(B, N, -1)
    return sanitized

In [ ]:
import torch
from util.misc import nested_tensor_from_tensor_list

for epoch in range(20):
    mod.train()
    criterion.train()
    running_loss = 0.0
    skipped = 0
    
    for i, (samples, targets) in enumerate(data_loader_train):
        if any(len(t.get("boxes", [])) == 0 for t in targets):
            skipped += 1
            continue

        samples = nested_tensor_from_tensor_list(samples).to("cuda")
        targets = [{k: v.to("cuda") for k, v in t.items()} for t in targets]

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = mod(samples)
            outputs['pred_logits'] = torch.nan_to_num(outputs['pred_logits'], nan=0.0)
            outputs['pred_boxes'] = sanitize_boxes_cxcywh(outputs['pred_boxes'])

            if 'aux_outputs' in outputs:
                for aux in outputs['aux_outputs']:
                    aux['pred_logits'] = torch.nan_to_num(aux['pred_logits'], nan=0.0)
                    aux['pred_boxes'] = sanitize_boxes_cxcywh(aux['pred_boxes'])

            try:
                loss_dict = criterion(outputs, targets)
                weight_dict = criterion.weight_dict
                losses = sum(loss_dict[k] * weight_dict[k] 
                             for k in loss_dict.keys() if k in weight_dict)
            except ValueError as e:
                print(f"Matrix error at Iter {i}: {e}. Skipping batch.")
                continue

        if torch.isfinite(losses):
            scaler.scale(losses).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(mod.parameters(), max_norm=0.1)
            scaler.step(optimizer)
            scaler.update()
            running_loss += losses.item()
        else:
            print(f"Loss is NaN at Iter {i}, skipping backward pass.")

        if (i + 1) % 100 == 0:
            print(f"Epoch {epoch+1}, Iter {i+1}, Loss: {losses.item():.4f}")

    avg_loss = running_loss / max(len(data_loader_train) - skipped, 1)
    print(f"Epoch {epoch+1:2d} | Avg Loss: {avg_loss:.4f} | Skipped: {skipped}")
    if (epoch + 1) % 2 == 0:
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': mod.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'loss': avg_loss,
        }
        torch.save(checkpoint, f"checkpoint_epoch_{epoch+1}.pth")
        print(f"Saved checkpoint at epoch {epoch+1}")

In [ ]:
mod.eval()
for module in mod.modules():
    if isinstance(module, Ternary):
        module.pack_for_inference()
 
torch.save(mod.state_dict(), "/kaggle/working/detr_1.58bit_packed_final.pth")